# 规则引擎示例

本notebook展示如何使用hscredit的规则引擎功能，包括规则定义、逻辑运算、表达式优化和效果评估。

In [1]:
import sys
sys.path.insert(0, '/Users/xiaoxi/CodeBuddy/hscredit/hscredit')


import pandas as pd
import numpy as np
from hscredit import Rule, optimize_expr, beautify_expr, get_expr_variables

## 1. 加载数据

In [2]:
# 加载示例数据
df = pd.read_excel('/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/hscredit.xlsx')
print(f"数据形状: {df.shape}")
print(f"列名: {df.columns.tolist()}")

# 创建目标变量: MOB1 > 15 表示账龄超过15个月
df['target'] = (df['MOB1'] > 15).astype(int)
print(f"\n目标变量分布:")
print(df['target'].value_counts())

数据形状: (22729, 12)
列名: ['MOB1', 'MOB2', '青云24', '游昆定制分80', '百融定制分V9', '中智小牛分C3', '度小满欺诈因子V6PRO多头版', '百行百川分FPV1', '游昆设备黑名单', '业务类型', '流量渠道', '放款期数']

目标变量分布:
target
0    21122
1     1607
Name: count, dtype: int64


## 2. 定义基础规则

In [3]:
# 使用数据中的数值型变量创建规则
# 规则表达式使用pandas eval语法

# 规则1: 青云24分数大于700
rule_qy24 = Rule('青云24 > 700')

# 规则2: 游昆定制分大于750
rule_yk_score = Rule('游昆定制分80 > 750')

# 规则3: 百融定制分大于某个阈值
rule_br_score = Rule('百融定制分V9 > 650')

# 规则4: 设备黑名单次数大于0
rule_blacklist = Rule('游昆设备黑名单 > 0')

print("基础规则定义:")
print(f"rule_qy24: {rule_qy24.expr}")
print(f"rule_yk_score: {rule_yk_score.expr}")
print(f"rule_br_score: {rule_br_score.expr}")
print(f"rule_blacklist: {rule_blacklist.expr}")

基础规则定义:
rule_qy24: 青云24 > 700
rule_yk_score: 游昆定制分80 > 750
rule_br_score: 百融定制分V9 > 650
rule_blacklist: 游昆设备黑名单 > 0


## 3. 规则逻辑运算与表达式优化

In [4]:
print("=" * 60)
print("3.1 幂等律 (Idempotent Law): A & A = A, A | A = A")
print("=" * 60)
r1 = Rule('青云24 > 700')
result = r1 & r1
print(f"r1 & r1: {result.expr}")

result = r1 | r1
print(f"r1 | r1: {result.expr}")

3.1 幂等律 (Idempotent Law): A & A = A, A | A = A
r1 & r1: 青云24 > 700
r1 | r1: 青云24 > 700


In [5]:
print("=" * 60)
print("3.2 双重否定 (Double Negation): ~~A = A")
print("=" * 60)
r1 = Rule('青云24 > 700')
result = ~~r1
print(f"~~r1: {result.expr}")

3.2 双重否定 (Double Negation): ~~A = A
~~r1: 青云24 > 700


In [6]:
print("=" * 60)
print("3.3 吸收率 (Absorption Law): A | (A & B) = A")
print("=" * 60)
r1 = Rule('青云24 > 700')
r2 = Rule('游昆定制分80 > 750')

# A | (A & B) = A
result = r1 | (r1 & r2)
print(f"r1 | (r1 & r2): {result.expr}")

# (A & B) | B = B
result = (r1 & r2) | r2
print(f"(r1 & r2) | r2: {result.expr}")

# A & (A | B) = A
result = r1 & (r1 | r2)
print(f"r1 & (r1 | r2): {result.expr}")

3.3 吸收率 (Absorption Law): A | (A & B) = A
r1 | (r1 & r2): 青云24 > 700 | (青云24 > 700 & 游昆定制分80 > 750)
(r1 & r2) | r2: 游昆定制分80 > 750
r1 & (r1 | r2): 青云24 > 700 | 游昆定制分80 > 750


In [7]:
print("=" * 60)
print("3.4 基本逻辑运算")
print("=" * 60)
r1 = Rule('青云24 > 700')
r2 = Rule('游昆定制分80 > 750')

# 与运算 (AND)
result = r1 & r2
print(f"r1 & r2 (与): {result.expr}")

# 或运算 (OR)
result = r1 | r2
print(f"r1 | r2 (或): {result.expr}")

# 异或运算 (XOR)
result = r1 ^ r2
print(f"r1 ^ r2 (异或): {result.expr}")

# 非运算 (NOT)
result = ~r1
print(f"~r1 (非): {result.expr}")

3.4 基本逻辑运算
r1 & r2 (与): 青云24 > 700 & 游昆定制分80 > 750
r1 | r2 (或): 青云24 > 700 | 游昆定制分80 > 750
r1 ^ r2 (异或): 青云24 > 700 ^ 游昆定制分80 > 750
~r1 (非): ~(青云24 > 700)


In [8]:
print("=" * 60)
print("3.5 复杂组合规则")
print("=" * 60)
r1 = Rule('青云24 > 700')
r2 = Rule('游昆定制分80 > 750')
r3 = Rule('百融定制分V9 > 650')
r4 = Rule('游昆设备黑名单 > 0')

# 组合: (规则1 与 规则2) 或 规则3
complex_rule = (r1 & r2) | r3
print(f"(r1 & r2) | r3: {complex_rule.expr}")

# 更复杂的组合
complex_rule = (r1 & r2) | (r3 & ~r4)
print(f"(r1 & r2) | (r3 & ~r4): {complex_rule.expr}")

3.5 复杂组合规则
(r1 & r2) | r3: (青云24 > 700 & 游昆定制分80 > 750) | 百融定制分V9 > 650
(r1 & r2) | (r3 & ~r4): (青云24 > 700 & 游昆定制分80 > 750) | (百融定制分V9 > 650 & ~(游昆设备黑名单 > 0))


## 4. 优化器函数

In [9]:
# 优化表达式: 应用布尔代数定律简化
print("=" * 60)
print("4.1 optimize_expr - 简化表达式")
print("=" * 60)

expr1 = '(青云24 > 700) & (青云24 > 700)'
print(f"优化前: {expr1}")
print(f"优化后: {optimize_expr(expr1)}")

expr2 = '~~(青云24 > 700)'
print(f"\n优化前: {expr2}")
print(f"优化后: {optimize_expr(expr2)}")

expr3 = '(青云24 > 700 & 游昆定制分80 > 750) | 游昆定制分80 > 750'
print(f"\n优化前: {expr3}")
print(f"优化后: {optimize_expr(expr3)}")

4.1 optimize_expr - 简化表达式
优化前: (青云24 > 700) & (青云24 > 700)
优化后: 青云24 > 700

优化前: ~~(青云24 > 700)
优化后: 青云24 > 700

优化前: (青云24 > 700 & 游昆定制分80 > 750) | 游昆定制分80 > 750
优化后: 青云24 > 700 & 游昆定制分80 > 750 | 游昆定制分80 > 750


In [10]:
print("=" * 60)
print("4.2 beautify_expr - 美化表达式格式")
print("=" * 60)

expr = '(青云24 > 700) & (游昆定制分80 > 750)'
print(f"美化前: {expr}")
print(f"美化后: {beautify_expr(expr)}")

4.2 beautify_expr - 美化表达式格式
美化前: (青云24 > 700) & (游昆定制分80 > 750)
美化后: 青云24 > 700 & 游昆定制分80 > 750


In [11]:
print("=" * 60)
print("4.3 get_expr_variables - 提取变量名")
print("=" * 60)

expr = '(青云24 > 700) & (游昆定制分80 > 750)'
print(f"表达式: {expr}")
print(f"变量: {get_expr_variables(expr)}")

4.3 get_expr_variables - 提取变量名
表达式: (青云24 > 700) & (游昆定制分80 > 750)
变量: []


## 5. 规则效果评估

In [12]:
# 使用Rule的report方法评估规则效果
print("=" * 60)
print("5.1 单规则效果评估")
print("=" * 60)

# 定义规则: 青云24分数大于700
rule1 = Rule('青云24 > 700')

# 生成规则报告
report = rule1.report(df, target='target', desc='青云24分数>700')
print("规则报告:")
report

5.1 单规则效果评估
规则报告:


,规则分类,指标名称,指标含义,分箱,样本总数,样本占比,好样本数,好样本占比,坏样本数,坏样本占比,坏样本率,LIFT值,坏账改善,风险拒绝比,准确率,精确率,召回率,F1分数
0,验证规则,青云24 > 700,青云24分数>700,命中,1550.0,0.068195,1497.0,0.065863,53.0,0.002332,0.034194,0.483625,-0.035214,-0.516375,0.931805,0.0,0.0,0.0
1,验证规则,青云24 > 700,青云24分数>700,未命中,21179.0,0.931805,19625.0,0.863434,1554.0,0.068371,0.073375,1.037791,0.035214,0.037791,0.068195,0.0,0.0,0.0


In [13]:
# 定义规则: 游昆定制分大于750
rule2 = Rule('游昆定制分80 > 750')

report2 = rule2.report(df, target='target', desc='游昆定制分>750')
print("规则报告:")
print(report2)

规则报告:
   规则分类           指标名称       指标含义   分箱     样本总数  样本占比     好样本数     好样本占比  \
0  验证规则  游昆定制分80 > 750  游昆定制分>750   命中      0.0   0.0      0.0  0.000000   
1  验证规则  游昆定制分80 > 750  游昆定制分>750  未命中  22729.0   1.0  21122.0  0.929297   

     坏样本数     坏样本占比      坏样本率  LIFT值  坏账改善  风险拒绝比  准确率  精确率  召回率  F1分数  
0     0.0  0.000000  0.000000    0.0   0.0    NaN  1.0  0.0  0.0   0.0  
1  1607.0  0.070703  0.070703    1.0   0.0    0.0  0.0  0.0  0.0   0.0  


In [14]:
print("=" * 60)
print("5.2 组合规则效果评估")
print("=" * 60)

# 组合规则: 青云24 > 700 或 游昆定制分 > 750
combined_rule = rule1 | rule2

report_combined = combined_rule.report(df, target='target', desc='青云24>700 或 游昆定制分>750')
print("组合规则报告:")
print(report_combined)

5.2 组合规则效果评估
组合规则报告:
   规则分类                        指标名称                  指标含义   分箱     样本总数  \
0  验证规则  青云24 > 700 | 游昆定制分80 > 750  青云24>700 或 游昆定制分>750   命中   1550.0   
1  验证规则  青云24 > 700 | 游昆定制分80 > 750  青云24>700 或 游昆定制分>750  未命中  21179.0   

       样本占比     好样本数     好样本占比    坏样本数     坏样本占比      坏样本率     LIFT值  \
0  0.068195   1497.0  0.065863    53.0  0.002332  0.034194  0.483625   
1  0.931805  19625.0  0.863434  1554.0  0.068371  0.073375  1.037791   

       坏账改善     风险拒绝比       准确率  精确率  召回率  F1分数  
0 -0.035214 -0.516375  0.931805  0.0  0.0   0.0  
1  0.035214  0.037791  0.068195  0.0  0.0   0.0  


In [15]:
# 组合规则: 青云24 > 700 且 游昆定制分 > 750
combined_rule_and = rule1 & rule2

report_and = combined_rule_and.report(df, target='target', desc='青云24>700 且 游昆定制分>750')
print("组合规则(与)报告:")
print(report_and)

组合规则(与)报告:
   规则分类                        指标名称                  指标含义   分箱     样本总数  样本占比  \
0  验证规则  青云24 > 700 & 游昆定制分80 > 750  青云24>700 且 游昆定制分>750   命中      0.0   0.0   
1  验证规则  青云24 > 700 & 游昆定制分80 > 750  青云24>700 且 游昆定制分>750  未命中  22729.0   1.0   

      好样本数     好样本占比    坏样本数     坏样本占比      坏样本率  LIFT值  坏账改善  风险拒绝比  准确率  \
0      0.0  0.000000     0.0  0.000000  0.000000    0.0   0.0    NaN  1.0   
1  21122.0  0.929297  1607.0  0.070703  0.070703    1.0   0.0    0.0  0.0   

   精确率  召回率  F1分数  
0  0.0  0.0   0.0  
1  0.0  0.0   0.0  


## 6. 使用overdue参数

In [16]:
# 使用overdue参数进行逾期分析
print("=" * 60)
print("6.1 使用MOB1作为逾期天数分析")
print("=" * 60)

# 使用MOB1作为逾期天数字段
rule_mob = Rule('青云24 > 700')

# 设置overdue参数: MOB1 > 15 为坏账
report_mob = rule_mob.report(df, overdue='MOB1', dpds=15, desc='青云24分数>700')
print("基于MOB1>15的规则报告:")
print(report_mob)

6.1 使用MOB1作为逾期天数分析
基于MOB1>15的规则报告:
   规则分类        指标名称        指标含义   分箱     样本总数      样本占比     好样本数     好样本占比  \
0  验证规则  青云24 > 700  青云24分数>700   命中   1550.0  0.068195   1497.0  0.065863   
1  验证规则  青云24 > 700  青云24分数>700  未命中  21179.0  0.931805  19625.0  0.863434   

     坏样本数     坏样本占比      坏样本率     LIFT值      坏账改善     风险拒绝比       准确率  精确率  \
0    53.0  0.002332  0.034194  0.483625 -0.035214 -0.516375  0.931805  0.0   
1  1554.0  0.068371  0.073375  1.037791  0.035214  0.037791  0.068195  0.0   

   召回率  F1分数  
0  0.0   0.0  
1  0.0   0.0  


In [17]:
print("=" * 60)
print("6.2 使用MOB2作为逾期天数分析")
print("=" * 60)

report_mob2 = rule_mob.report(df, overdue='MOB2', dpds=15, desc='青云24分数>700')
print("基于MOB2>15的规则报告:")
print(report_mob2)

6.2 使用MOB2作为逾期天数分析
基于MOB2>15的规则报告:
   规则分类        指标名称        指标含义   分箱     样本总数      样本占比     好样本数     好样本占比  \
0  验证规则  青云24 > 700  青云24分数>700   命中   1550.0  0.068195   1472.0  0.064763   
1  验证规则  青云24 > 700  青云24分数>700  未命中  21179.0  0.931805  18752.0  0.825025   

     坏样本数     坏样本占比      坏样本率     LIFT值      坏账改善     风险拒绝比       准确率  精确率  \
0    78.0  0.003432  0.050323  0.456600 -0.037057 -0.543400  0.931805  0.0   
1  2427.0  0.106780  0.114595  1.039769  0.037057  0.039769  0.068195  0.0   

   召回率  F1分数  
0  0.0   0.0  
1  0.0   0.0  


## 7. 规则集评估

In [18]:
from hscredit import ruleset_report

# 定义多个规则
rules = [
    Rule('青云24 > 700'),
    Rule('游昆定制分80 > 750'),
    Rule('百融定制分V9 > 650'),
]

# 生成规则集报告
print("=" * 60)
print("7. 规则集报告")
print("=" * 60)

ruleset = ruleset_report(df, rules, target='target')
print("规则集报告:")
print(ruleset)

7. 规则集报告
规则集报告:
                 规则顺序  规则分类                                        指标名称 指标含义  \
0     规则1: 青云24 > 700  验证规则                                  青云24 > 700        
1     规则1: 青云24 > 700  验证规则                                  青云24 > 700        
2     规则1: 青云24 > 700   NaN                                         NaN  NaN   
3  规则2: 游昆定制分80 > 750  验证规则                               游昆定制分80 > 750        
4  规则2: 游昆定制分80 > 750  验证规则                               游昆定制分80 > 750        
5  规则2: 游昆定制分80 > 750   NaN                                         NaN  NaN   
6  规则3: 百融定制分V9 > 650  验证规则                               百融定制分V9 > 650        
7  规则3: 百融定制分V9 > 650  验证规则                               百融定制分V9 > 650        
8  规则3: 百融定制分V9 > 650   NaN                                         NaN  NaN   
9              全部规则汇总  验证规则  青云24 > 700 | 游昆定制分80 > 750 | 百融定制分V9 > 650        

     分箱     样本总数      样本占比     好样本数     好样本占比    坏样本数     坏样本占比      坏样本率  \
0    命中   1550.0  0.068195

In [22]:
ruleset

,规则顺序,规则分类,指标名称,指标含义,分箱,样本总数,样本占比,好样本数,好样本占比,坏样本数,坏样本占比,坏样本率,LIFT值,坏账改善,风险拒绝比,准确率,精确率,召回率,F1分数
0,规则1: 青云24 > 700,验证规则,青云24 > 700,,命中,1550.0,0.068195,1497.0,0.065863,53.0,0.002332,0.034194,0.483625,-0.035214,-0.516375,0.931805,0.0,0.0,0.0
1,规则1: 青云24 > 700,验证规则,青云24 > 700,,未命中,21179.0,0.931805,19625.0,0.863434,1554.0,0.068371,0.073375,1.037791,0.035214,0.037791,0.068195,0.0,0.0,0.0
2,规则1: 青云24 > 700,NaN,NaN,NaN,剩余样本,21179.0,0.931805,19625.0,NaN,1554.0,NaN,0.073375,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,规则2: 游昆定制分80 > 750,验证规则,游昆定制分80 > 750,,命中,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,NaN,1.000000,0.0,0.0,0.0
4,规则2: 游昆定制分80 > 750,验证规则,游昆定制分80 > 750,,未命中,22729.0,1.000000,21122.0,0.929297,1607.0,0.070703,0.070703,1.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0
5,规则2: 游昆定制分80 > 750,NaN,NaN,NaN,剩余样本,21179.0,0.931805,19625.0,NaN,1554.0,NaN,0.073375,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,规则3: 百融定制分V9 > 650,验证规则,百融定制分V9 > 650,,命中,19665.0,0.865194,18413.0,0.810110,1252.0,0.055084,0.063666,0.900482,-0.086103,-0.099518,0.134806,0.0,0.0,0.0
7,规则3: 百融定制分V9 > 650,验证规则,百融定制分V9 > 650,,未命中,3064.0,0.134806,2709.0,0.119187,355.0,0.015619,0.115862,1.638717,0.086103,0.638717,0.865194,0.0,0.0,0.0
8,规则3: 百融定制分V9 > 650,NaN,NaN,NaN,剩余样本,2999.0,0.131946,2650.0,NaN,349.0,NaN,0.116372,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,全部规则汇总,验证规则,青云24 > 700 | 游昆定制分80 > 750 | 百融定制分V9 > 650,,命中,19730.0,0.868054,18472.0,0.812706,1258.0,0.055348,0.063761,0.901816,-0.085229,-0.098184,0.131946,0.0,0.0,0.0


## 8. 规则预测

In [23]:
# 使用规则进行预测
print("=" * 60)
print("8. 规则预测")
print("=" * 60)

rule = Rule('青云24 > 700')

# 预测: 返回布尔Series
predictions = rule.predict(df)
print(f"预测结果类型: {type(predictions)}")
print(f"命中样本数: {predictions.sum()}")
print(f"未命中样本数: {(~predictions).sum()}")

# 过滤数据
matched_df = rule.filter(df)
print(f"\n规则命中的数据形状: {matched_df.shape}")

8. 规则预测
预测结果类型: <class 'pandas.core.series.Series'>
命中样本数: 1550
未命中样本数: 21179

规则命中的数据形状: (1550, 13)


## 9. 总结

### 规则引擎主要功能:

1. **规则定义**: 使用`Rule(expr)`定义规则表达式

2. **逻辑运算**: 支持`&` (与), `|` (或), `^` (异或), `~` (非)

3. **表达式优化**: 
   - `optimize_expr()`: 应用布尔代数定律简化表达式
   - `beautify_expr()`: 美化表达式格式
   - `get_expr_variables()`: 提取变量名

4. **规则评估**: 
   - `Rule.report()`: 单规则效果评估
   - `ruleset_report()`: 多规则综合评估

5. **规则预测**: 
   - `Rule.predict()`: 应用规则预测
   - `Rule.filter()`: 根据规则过滤数据